# 00 Validação do ambiente Databricks
Datathon Fase 5 | Passos Mágicos

Execute as células em ordem no workspace. Preencha catálogo e schema existentes na célula 2. A execução cria uma tabela Delta e um modelo de teste com nomes exclusivos e registra um run MLflow. Use somente dados sintéticos. O endpoint é opcional e sua criação é manual.

## 1. Compute e dependências
A primeira consulta deve retornar três linhas. Se um import falhar, confira o ambiente Python do notebook e instale apenas a dependência ausente antes de reexecutar.

In [0]:
display(spark.range(3))
import sys
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import pandas as pd
import sklearn

print("Python", sys.version.split()[0])
print("MLflow", mlflow.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
display(spark.sql("SHOW CATALOGS"))

## 2. Configuração
Confira em **Catalog** o schema autorizado. Execute esta configuração uma vez por rodada e mantenha os identificadores ao retomar uma etapa.

In [0]:
import re
from uuid import uuid4

catalogo = "workspace"  # Preencha com o catálogo autorizado.
schema = "default"    # Preencha com o schema autorizado.
for nome in (catalogo, schema):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", nome):
        raise ValueError("Preencha catálogo e schema com identificadores simples.")

teste_id = uuid4().hex[:12]
tabela = f"{catalogo}.{schema}.fase5_teste_delta_{teste_id}"
nome_modelo = f"{catalogo}.{schema}.fase5_teste_modelo_{teste_id}"
volume = "vol_trio_drive"  # Opcional. Preencha com o nome de um volume existente.
print("Destino", f"{catalogo}.{schema}")
print("Rodada", teste_id)

### Arquivos de origem (opcional)
Se o grupo for usar um volume, envie pelo Catalog um arquivo `teste_fase5.csv` com cabeçalho `x` e três linhas contendo os valores 1, 2 e 3. Preencha o volume na configuração antes de executar a rodada.

In [0]:
if volume:
    caminho = (
    f"/Volumes/{catalogo}/{schema}/{volume}/"
    "projetos/fiap/datathon_fase5/teste_fase5.csv"
)
    arquivo = spark.read.option("header", True).csv(caminho)
    assert arquivo.columns == ["x"] and arquivo.count() == 3
    print("Leitura do CSV aprovada", caminho)
else:
    print("Teste de volume não executado.")

## 3. Escrita e leitura Delta
Se esta célula já passou, siga para a próxima. Para repetir a escrita com um objeto novo, inicie outra rodada pela configuração.

In [0]:
spark.sql(f"""
    CREATE TABLE {tabela}
    USING DELTA
    AS SELECT id FROM range(3)
""")

resultado = spark.table(tabela).orderBy("id")
assert [linha.id for linha in resultado.collect()] == [0, 1, 2]

display(resultado)
print("Delta aprovado", tabela)

## 4. Experimento MLflow
Treine uma regressão sintética com y = 2 × x. Confira parâmetro, métrica e modelo no run gerado. Use o experimento associado ao notebook ou preencha o caminho de um experimento autorizado.

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd

from mlflow.models import infer_signature
from sklearn.linear_model import LinearRegression

experimento = ""  # Opcional. Caminho absoluto no workspace.
mlflow.set_tracking_uri("databricks")
if experimento:
    mlflow.set_experiment(experimento)

X = pd.DataFrame({"x": [1.0, 2.0, 3.0, 4.0]})
y = pd.Series([2.0, 4.0, 6.0, 8.0])
modelo = LinearRegression().fit(X, y)
estimativas = modelo.predict(X)
assinatura = infer_signature(X, estimativas)
argumento_nome = (
    {"name": "modelo"} if int(mlflow.__version__.split(".")[0]) >= 3
    else {"artifact_path": "modelo"}
)

with mlflow.start_run(run_name=f"fase5_validacao_{teste_id}") as run:
    mlflow.log_param("origem", "dados_sinteticos")
    mlflow.log_metric("mae_treino", float(abs(estimativas - y).mean()))
    info = mlflow.sklearn.log_model(
        sk_model=modelo, signature=assinatura,
        input_example=X.iloc[:1], **argumento_nome,
    )
    run_id = run.info.run_id
print("Run ID", run_id)
print("Modelo no MLflow", info.model_uri)

## 5. Registro no Unity Catalog
Confira a versão e a assinatura em **Catalog**. Se a chamada falhar, confira as permissões e verifique se já surgiu uma versão antes de repetir o registro.

In [0]:
mlflow.set_registry_uri("databricks-uc")
registro = mlflow.register_model(info.model_uri, nome_modelo)
uri_versao = f"models:/{nome_modelo}/{registro.version}"
print("Versão registrada", uri_versao)

## 6. Inferência válida e inválida
Uma segunda pessoa pode repetir esta célula com a própria conta após configurar o registry e informar o URI exato da versão autorizada recebido do grupo.

In [0]:
from math import isclose
from mlflow.exceptions import MlflowException

modelo_registrado = mlflow.pyfunc.load_model(uri_versao)
saida = modelo_registrado.predict(pd.DataFrame({"x": [5.0]}))
previsao = float(saida[0])
assert isclose(previsao, 10.0, rel_tol=0.0, abs_tol=1e-8)
print("Entrada válida aprovada", previsao)

try:
    modelo_registrado.predict(pd.DataFrame({"coluna_errada": [5.0]}))
except MlflowException as erro:
    if "missing" not in str(erro).lower() or "x" not in str(erro):
        raise
    print("Entrada inválida rejeitada", str(erro))
else:
    raise AssertionError("A entrada sem o campo x deveria ser rejeitada.")

## 7. Serving e evidências
Se o workspace permitir, crie manualmente um endpoint em **Serving** apontando para a versão acima. Quando estiver pronto, teste no painel de consulta.

- Entrada válida: `{"dataframe_records": [{"x": 5.0}]}`. Previsão esperada: aproximadamente 10.
- Entrada inválida: `{"dataframe_records": [{"coluna_errada": 5.0}]}`. Registre a rejeição.
- Se Serving estiver indisponível, registre o motivo e use a inferência via notebook da célula anterior. Confirme acesso de uma segunda pessoa com a própria conta.
- Valide também GitHub e o acesso da equipe conforme o guia Word. Registre o commit, a rota escolhida e o resultado de cada teste.

In [0]:
print({
    "tabela_delta": tabela,
    "run_id": run_id,
    "modelo_registrado": uri_versao,
    "previsao_sintetica": previsao,
    "inferencia_validada": "via notebook",
})